# CampusHelp: Student FAQ Semantic Search

This notebook loads FAQ data, cleans it, stores the FAQ text in Chroma, and searches for answers to a question.





## 1. Load the FAQ data


## 2. Clean the data


In [7]:
from google.colab import files

uploaded = files.upload()

Saving faqs.csv to faqs.csv


In [8]:
from IPython.display import display
import pandas as pd

df = pd.read_csv("faqs.csv")
print("Number of FAQs:", len(df))
display(df.head())

Number of FAQs: 10


,id,category,question,answer
0,1,Registration,How do I register for courses?,Log in to the student portal and follow your d...
1,2,Registration,What if I miss the course registration deadline?,Contact your department or registrar to ask wh...
2,3,Library,When can I use the university library?,Check the library's published opening hours fo...
3,4,Library,Can I find a quiet place to study?,The university library may provide quiet study...
4,5,Exams,How can I check my exam results?,Check the student portal or follow the officia...


In [9]:
df = df.dropna(subset=["question", "answer"])

for col in ["category", "question", "answer"]:
    df[col] = df[col].astype(str).str.strip()

df = df[(df["question"] != "") & (df["answer"] != "")]
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)

print("FAQs after cleaning:", len(df))
display(df.head())

FAQs after cleaning: 10


,id,category,question,answer
0,1,Registration,How do I register for courses?,Log in to the student portal and follow your d...
1,2,Registration,What if I miss the course registration deadline?,Contact your department or registrar to ask wh...
2,3,Library,When can I use the university library?,Check the library's published opening hours fo...
3,4,Library,Can I find a quiet place to study?,The university library may provide quiet study...
4,5,Exams,How can I check my exam results?,Check the student portal or follow the officia...


## 3. Make text chunks

Each FAQ is short, so I use one chunk for each FAQ. This keeps the question and its answer together.


In [10]:
chunks = []

for _, row in df.iterrows():
    text = (
        "Category: " + row["category"] + "\n" +
        "Question: " + row["question"] + "\n" +
        "Answer: " + row["answer"]
    )
    chunks.append({
        "id": str(row["id"]),
        "category": row["category"],
        "question": row["question"],
        "text": text
    })

print("Chunks created:", len(chunks))
print(chunks[0]["text"])

Chunks created: 10
Category: Registration
Question: How do I register for courses?
Answer: Log in to the student portal and follow your department's registration instructions.


## 4. Store the chunks in Chroma

Chroma uses an embedding model to turn text into vectors. On the first run, its default model may need to download files, so this step needs an internet connection if the model is not already installed.


In [12]:
%pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 63.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Suc

## 5. Search for a question

Try changing the question below. Chroma compares its embedding with the stored FAQ embeddings and returns the closest matches.


In [16]:
def search_faq(question, number_of_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=min(number_of_results, collection.count())
    )

    for i, text in enumerate(results["documents"][0]):
        print("Result", i + 1)
        print("Question:", results["metadatas"][0][i]["question"])
        print("Category:", results["metadatas"][0][i]["category"])
        print("FAQ text:", text)
        print("-" * 40)

search_faq("How can I change my student portal password?")

Result 1
Question: How can I reset my student portal password?
Category: Accounts
FAQ text: Category: Accounts
Question: How can I reset my student portal password?
Answer: Use the portal's password recovery option if available or contact the appropriate IT support team.
----------------------------------------
Result 2
Question: What if I cannot log in to my student account?
Category: Accounts
FAQ text: Category: Accounts
Question: What if I cannot log in to my student account?
Answer: Check your login details and contact the university's IT support if the problem continues.
----------------------------------------
Result 3
Question: How do I register for courses?
Category: Registration
FAQ text: Category: Registration
Question: How do I register for courses?
Answer: Log in to the student portal and follow your department's registration instructions.
----------------------------------------


## 6. Try other queries


In [14]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")
embedding_function = embedding_functions.DefaultEmbeddingFunction()
# Recreate the collection so old test data does not mix with this run.
try:
    client.delete_collection("campus_faqs")
except Exception:
    pass

collection = client.create_collection(
    name="campus_faqs",
    embedding_function=embedding_function
)
documents = [item["text"] for item in chunks]
ids = ["faq_" + item["id"] for item in chunks]
metadatas = [
    {"category": item["category"], "question": item["question"]}
    for item in chunks
]

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print("FAQs stored in Chroma:", collection.count())

FAQs stored in Chroma: 10


In [17]:
search_faq("I missed the deadline to choose my courses")
search_faq("Where do I see my exam marks?")
search_faq("Where can I study quietly?")

Result 1
Question: What if I miss the course registration deadline?
Category: Registration
FAQ text: Category: Registration
Question: What if I miss the course registration deadline?
Answer: Contact your department or registrar to ask whether late registration is permitted.
----------------------------------------
Result 2
Question: How do I register for courses?
Category: Registration
FAQ text: Category: Registration
Question: How do I register for courses?
Answer: Log in to the student portal and follow your department's registration instructions.
----------------------------------------
Result 3
Question: What should I do if I miss an examination?
Category: Exams
FAQ text: Category: Exams
Question: What should I do if I miss an examination?
Answer: Contact your instructor or department as soon as possible to ask about the official procedure.
----------------------------------------
Result 1
Question: How can I check my exam results?
Category: Exams
FAQ text: Category: Exams
Question